In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_H.xpt"
csv_file = DATA_DIR / "HUQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 HUQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ041",
    "HUQ051",
    "HUQ061",
    "HUQ071",
    "HUD080",
    "HUQ090"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HUQ_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 10175:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,175 rows x 10 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# HUQ_H contains no genuine continuous decimal variables.
#
# All fields are:
#
#     respondent ID
#     questionnaire codes
#     whole-number count categories
#
# Therefore all 10 fields can ultimately become Int64.
#
# IMPORTANT:
#
# This applies only to HUQ_H.
#
# Genuine Dietary, Laboratory, Examination,
# sample-weight, and other true decimal fields in
# other NHANES files should still remain untouched.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 10:

    raise ValueError(
        "Expected exactly 10 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER blanket-replace this value before first
# confirming its exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 HUQ_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached HUQ_H.xpt:
#
# HUQ051 = 1,287 tiny values
#
# TOTAL  = 1,287
#
# CDC:
#
# HUQ051 = 0
#     None
#     Count = 1,287
#

EXPECTED_TINY_COUNTS = {
    "HUQ051": 1287
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated HUQ_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 1287:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "already existed before correction."
    )


print(
    "\nPASS: Exact HUQ051 1,287-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED HUQ051 ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only the known/validated HUQ051 values.
#

huq051_tiny_mask = (
    df["HUQ051"]
    .eq(TINY_VALUE)
)


correction_count = int(
    huq051_tiny_mask.sum()
)


if correction_count != 1287:

    raise ValueError(
        "Unexpected HUQ051 tiny-value count."
    )


df.loc[
    huq051_tiny_mask,
    "HUQ051"
] = 0


print("\n--- VALIDATED ZERO RESTORATION ---")


print(
    "HUQ051 restored to zero:",
    f"{correction_count:,}"
)


if int(
    df["HUQ051"]
    .eq(0)
    .sum()
) != 1287:

    raise ValueError(
        "HUQ051 zero restoration failed."
    )


print(
    "PASS: 1,287 validated HUQ051 zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# source should contain.
#
# The ONLY allowed numeric modification:
#
#     HUQ051 tiny -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["HUQ051"]
    .eq(TINY_VALUE),
    "HUQ051"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE converting anything to Int64.
#
# np.round() below is ONLY used as a test.
#
# It does NOT modify any data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in HUQ_H. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 12. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All 10 fields have now been proven whole-number.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 HUQ_H variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        "Observed:",
        actual_values
    )


    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. CDC CHECK - HUQ010
# ============================================================

#
# General health condition.
#

check_frequency(
    "HUQ010",
    {
        1: 2615,
        2: 2716,
        3: 3235,
        4: 1330,
        5: 274,
        7: 2,
        9: 3
    },
    0
)


# ============================================================
# 15. CDC CHECK - HUQ020
# ============================================================

#
# Health now compared with 1 year ago.
#
# Asked only for age >=1.
#

check_frequency(
    "HUQ020",
    {
        1: 1895,
        2: 760,
        3: 7112,
        7: 1,
        9: 1
    },
    406
)


# ============================================================
# 16. CDC CHECK - HUQ030
# ============================================================

#
# Routine place to go for healthcare.
#

check_frequency(
    "HUQ030",
    {
        1: 8944,
        2: 1194,
        3: 37
    },
    0
)


# ============================================================
# 17. CDC CHECK - HUQ041
# ============================================================

#
# Type of place most often used for healthcare.
#
# Populated when HUQ030 = 1 or 3.
#

check_frequency(
    "HUQ041",
    {
        1: 1813,
        2: 6735,
        3: 234,
        4: 90,
        5: 70,
        6: 38,
        77: 1
    },
    1194
)


# ============================================================
# 18. HUQ041 ROUTING CHECK
# ============================================================

#
# HUQ041 should be populated exactly when:
#
# HUQ030 = 1
#
# OR
#
# HUQ030 = 3
#

huq041_expected = (
    df["HUQ030"]
    .isin(
        [
            1,
            3
        ]
    )
    .to_numpy(
        dtype=bool
    )
)


huq041_present = (
    df["HUQ041"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


huq041_mismatch_count = int(
    np.sum(
        huq041_expected
        !=
        huq041_present
    )
)


print(
    "\n--- HUQ041 ROUTING ---"
)


print(
    "Routing mismatches:",
    huq041_mismatch_count
)


if huq041_mismatch_count != 0:

    raise ValueError(
        "HUQ041 routing does not match "
        "HUQ030 = 1 or 3."
    )


print(
    "PASS: HUQ041 routing validated."
)


# ============================================================
# 19. CDC CHECK - HUQ051
# ============================================================

#
# Number of healthcare visits over past year.
#
# CDC:
#
# 0 = None            1,287
# 1 = 1               2,167
# 2 = 2 to 3          3,206
# 3 = 4 to 5          1,518
# 4 = 6 to 7            705
# 5 = 8 to 9            289
# 6 = 10 to 12          460
# 7 = 13 to 15          171
# 8 = 16 or more        361
# 99 = Don't know        11
#
# Missing = 0
#

check_frequency(
    "HUQ051",
    {
        0: 1287,
        1: 2167,
        2: 3206,
        3: 1518,
        4: 705,
        5: 289,
        6: 460,
        7: 171,
        8: 361,
        99: 11
    },
    0
)


# ============================================================
# 20. CDC CHECK - HUQ061
# ============================================================

#
# How long since last healthcare visit.
#
# Asked only when HUQ051 = 0.
#

check_frequency(
    "HUQ061",
    {
        1: 71,
        2: 85,
        3: 521,
        4: 378,
        5: 204,
        6: 22,
        99: 6
    },
    8888
)


# ============================================================
# 21. HUQ051 / HUQ061 ROUTING CHECK
# ============================================================

#
# HUQ061 is asked exactly for respondents who
# reported HUQ051 = 0.
#

huq051_none = (
    df["HUQ051"]
    .eq(0)
    .to_numpy(
        dtype=bool
    )
)


huq061_present = (
    df["HUQ061"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


huq061_mismatch_count = int(
    np.sum(
        huq051_none
        !=
        huq061_present
    )
)


print(
    "\n--- HUQ051 / HUQ061 ROUTING ---"
)


print(
    "Routing mismatches:",
    huq061_mismatch_count
)


if huq061_mismatch_count != 0:

    raise ValueError(
        "HUQ061 routing does not exactly "
        "match HUQ051=0."
    )


print(
    "PASS: All 1,287 HUQ051=None records "
    "route exactly to HUQ061."
)


# ============================================================
# 22. CDC CHECK - HUQ071
# ============================================================

#
# Overnight hospital patient in last year.
#

check_frequency(
    "HUQ071",
    {
        1: 927,
        2: 9246,
        9: 2
    },
    0
)


# ============================================================
# 23. CDC CHECK - HUD080
# ============================================================

#
# Number of overnight hospital admissions.
#
# Asked when HUQ071 = 1.
#

check_frequency(
    "HUD080",
    {
        1: 706,
        2: 126,
        3: 55,
        4: 15,
        5: 9,
        6: 16
    },
    9248
)


# ============================================================
# 24. HUQ071 / HUD080 ROUTING CHECK
# ============================================================

huq071_yes = (
    df["HUQ071"]
    .eq(1)
    .to_numpy(
        dtype=bool
    )
)


hud080_present = (
    df["HUD080"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hud080_mismatch_count = int(
    np.sum(
        huq071_yes
        !=
        hud080_present
    )
)


print(
    "\n--- HUQ071 / HUD080 ROUTING ---"
)


print(
    "Routing mismatches:",
    hud080_mismatch_count
)


if hud080_mismatch_count != 0:

    raise ValueError(
        "HUD080 routing does not exactly "
        "match HUQ071=1."
    )


print(
    "PASS: All 927 overnight-hospital records "
    "route exactly to HUD080."
)


# ============================================================
# 25. CDC CHECK - HUQ090
# ============================================================

#
# Seen/talked to mental health professional
# during past 12 months.
#
# Asked age >=4.
#

check_frequency(
    "HUQ090",
    {
        1: 734,
        2: 8271,
        7: 1,
        9: 4
    },
    1165
)


# ============================================================
# 26. ALLOWED CODE CHECK
# ============================================================

ALLOWED_VALUES = {

    "HUQ010": {
        1, 2, 3, 4, 5, 7, 9
    },

    "HUQ020": {
        1, 2, 3, 7, 9
    },

    "HUQ030": {
        1, 2, 3, 7, 9
    },

    "HUQ041": {
        1, 2, 3, 4, 5, 6, 77, 99
    },

    "HUQ051": {
        0, 1, 2, 3, 4, 5, 6, 7, 8, 77, 99
    },

    "HUQ061": {
        1, 2, 3, 4, 5, 6, 77, 99
    },

    "HUQ071": {
        1, 2, 7, 9
    },

    "HUD080": {
        1, 2, 3, 4, 5, 6, 77777, 99999
    },

    "HUQ090": {
        1, 2, 7, 9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


    print(
        f"PASS: {col} contains only "
        "CDC-documented values."
    )


# ============================================================
# 27. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original file contained zero ordinary numeric zeros.
#
# After repair, the ONLY zero values should be:
#
#     HUQ051 = 1,287
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated HUQ_H correction pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern contains exactly "
    "1,287 HUQ051 zeros."
)


# ============================================================
# 28. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY permitted numeric modification:
#
#     HUQ051:
#     1,287 tiny values -> 0
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 10 variables match the exact "
    "expected corrected XPT values."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export.
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# HUQ_H contains no decimals, but retaining the same
# export method protects consistency across your full
# NHANES workflow.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 30. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 32. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 33. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 35. RAW CSV .0 CHECK
# ============================================================

#
# HUQ_H contains NO genuine decimal fields.
#
# Therefore it is safe to check all 10 fields
# for unnecessary ".0".
#
# Examples:
#
# SEQN
#     73557.0 -> 73557
#
# HUQ010
#     2.0 -> 2
#
# HUQ051
#     tiny -> 0
#
# HUD080
#     2.0 -> 2
#
# Do NOT use this global approach on mixed decimal files.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HUQ_H field."
)


# ============================================================
# 36. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "1,287 HUQ051 zeros."
)


# ============================================================
# 37. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 38. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 39. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "HUQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - "
    "Hospital Utilization & Access to Care"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "HUQ051 zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "HUQ051 None:",
    int(
        df["HUQ051"]
        .eq(0)
        .sum()
    )
)


print(
    "HUQ071 overnight hospital = Yes:",
    int(
        df["HUQ071"]
        .eq(1)
        .sum()
    )
)


print(
    "HUQ090 mental health professional = Yes:",
    int(
        df["HUQ090"]
        .eq(1)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 HUQ_H XPT -> CSV
Rows: 10,175
Columns: 10
PASS: Structure = 10,175 rows x 10 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Whole-number/code variables: 10

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 1,287

Tiny values by variable:
HUQ051    1287

PASS: Exact HUQ051 1,287-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
HUQ051 restored to zero: 1,287
PASS: 1,287 validated HUQ051 zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HUQ010: 0 fractional observations
HUQ020: 0 fractional observations
HUQ030: 0 fractional observations
HUQ041: 0 fractional observations
HUQ051: 0 fractional observations
HUQ061: 0 fractional observations
HUQ071: 0 fractional observations
HUD080: 0 fractional observations
HUQ090: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Mini